# Hermes mini：记忆、技能、检索与后台复盘

这个 Notebook 对应 `hermes_harness/`，重点不是单次问答，而是：

1. `MEMORY.md` / `USER.md` 的有界长期记忆；
2. `SKILL.md` 的程序性经验和渐进披露；
3. SQLite FTS5 会话检索；
4. 回合后的 background review；
5. 下一次新会话如何复用上一次学到的东西。

先确保项目根目录 `.env` 有百炼 Key。所有运行数据都写到临时目录，不会污染仓库。

In [ ]:
from pathlib import Path
import tempfile

from hermes_harness.memory import MemoryStore
from hermes_harness.skills import SkillStore
from hermes_harness.sessions import SessionStore

home = Path(tempfile.mkdtemp(prefix="hermes-notebook-"))
print("临时 Hermes home:", home)


## 1. 声明性记忆：MEMORY.md 与 USER.md

注意字符预算和 `<memory-context>` 围栏。

In [ ]:
memory = MemoryStore(home)
memory.add("memory", "这个项目使用 uv 管理依赖。")
memory.add("user", "用户希望回答先给结论，再给依据。")
print(memory.snapshot())
print(memory.context_block())
print(memory.stats())


## 2. 程序性记忆：SKILL.md

索引只展示 name/description/category，正文要 `view` 才加载。

In [ ]:
skills = SkillStore(home)
skills.save(
    "repo-research",
    "快速梳理一个代码仓库的结构与测试入口",
    "先读 README、pyproject 和测试目录，再给出模块地图、运行方式和风险。",
    category="research",
    files={"references/checklist.md": "README → 入口 → 测试 → 配置 → 风险"},
)
print(skills.index_text())
view = skills.view("repo-research")
print(view.content)
print("support files:", view.support_files)


## 3. 历史检索：SQLite FTS5 + 中文 bigram

In [ ]:
sessions = SessionStore(home)
session_id = sessions.create_session("第一次 Hermes 任务")
sessions.append_message(session_id, "user", "我们讨论了长期记忆和技能自动整理。")
sessions.append_message(session_id, "assistant", "后台复盘会把稳定偏好写入 USER.md。")
sessions.end_session(session_id)
print(sessions.search("长期记忆"))


## 4. 完整 Harness

需要真实模型调用。复用 `examples/hermes_harness_demo` 的副本，运行后查看：

- `.hermes/memories/MEMORY.md`
- `.hermes/memories/USER.md`
- `.hermes/skills/`
- `.hermes/runs/<timestamp>/events.jsonl`


In [ ]:
import shutil
from pathlib import Path
from hermes_harness.config import HermesHarnessConfig
from hermes_harness.harness import HermesHarness

workspace = Path(tempfile.mkdtemp(prefix="hermes-demo-"))
source = Path.cwd() / "examples" / "hermes_harness_demo"
shutil.copytree(source, workspace, dirs_exist_ok=True)

config = HermesHarnessConfig(
    workspace=workspace,
    toolsets=("file", "memory", "skills", "session_search"),
    enable_background_review=True,
    console=False,
)
harness = HermesHarness(config)
try:
    result = harness.run("阅读 research.md，提炼三条结论；以后请先给结论再给依据。")
finally:
    harness.close()

print(result.final_answer)
print("memory:", result.memory_stats)
print("skills used:", result.skills_used)
print("review:", result.review)


## 5. 查看运行轨迹与复盘产物

In [ ]:
print("run dir:", result.run_dir)
print((result.run_dir / "summary.md").read_text(encoding="utf-8"))
if (Path(workspace) / ".hermes" / "memories").exists():
    for path in sorted((Path(workspace) / ".hermes" / "memories").glob("*.md")):
        print(f"===== {path.name} =====")
        print(path.read_text(encoding="utf-8"))


## 6. 对照三个 Harness

- `mini_codex`：开发场景，`AGENTS.md` + 工具循环 + verify + events；
- `claude_harness`：开发场景，`CLAUDE.md` + permission + hooks + LangGraph；
- `hermes_harness`：通用任务，MEMORY/USER + Skills + background review + session_search。

查看源码级流程图：`docs/system-maps/hermes-harness.html`。